# 🎓 수학교육평가론 - 협력학습 MVP
## Human Student 1명 + AI Student 2명의 협업 학습 시스템

**구성요소:**
- 사용자(Human) 1명 + AI학생 2명(민준, 서연)
- Task 1~4: 수학 문제 (텍스트)
- 학습자 모델 4개: 인지 상태, 수학적 의사소통, 감정 상태, 협력 역량
- Claude API를 통한 AI 학생 발화 생성 및 학습자 모델 업데이트


In [ ]:
# 셀 1: 설치 및 API 설정
!pip install anthropic -q

import anthropic
import json
import copy
from datetime import datetime
from google.colab import userdata

API_KEY = userdata.get('CLAUDE_API_KEY')
client = anthropic.Anthropic(api_key=API_KEY)
MODEL = "claude-sonnet-4-20250514"

print("✅ API 연결 완료")


## 📊 학습자 모델 정의

4가지 학습자 모델을 정의합니다:
1. **인지 상태** — 개념이해도, 절차적 유창성, 오개념
2. **수학적 의사소통** — 표현 명확성, 논증 수준, 수학 용어 사용
3. **감정 상태** — 몰입도, 자신감, 좌절감
4. **협력 역량** — 기여도, 반응성, 도움 주고받기


In [ ]:
# 셀 2: 학습자 모델 스키마

LEARNER_MODEL_SCHEMA = {
    "cognitive_state": {
        "name": "인지 상태 (Cognitive State)",
        "description": "학습자가 현재 과제의 수학적 개념을 얼마나 이해하고 있는지를 나타내는 모델",
        "dimensions": {
            "conceptual_understanding": {
                "name": "개념 이해도",
                "description": "핵심 수학 개념에 대한 이해 수준",
                "scale": "1-5 (1: 개념 미형성, 2: 부분적 이해, 3: 기본 이해, 4: 심화 이해, 5: 전이 가능한 이해)",
                "initial_value": 3
            },
            "procedural_fluency": {
                "name": "절차적 유창성",
                "description": "수학적 절차와 알고리즘을 정확하고 효율적으로 수행하는 능력",
                "scale": "1-5 (1: 절차 미숙지, 2: 부분 수행, 3: 기본 수행, 4: 유창한 수행, 5: 유연한 적용)",
                "initial_value": 3
            },
            "misconceptions": {
                "name": "오개념 목록",
                "description": "관찰된 수학적 오개념들의 기록",
                "scale": "문자열 리스트",
                "initial_value": []
            }
        }
    },
    "math_communication": {
        "name": "수학적 의사소통 능력 (Mathematical Communication)",
        "description": "수학적 아이디어를 표현하고, 타인의 수학적 주장을 이해하며, 논증할 수 있는 능력",
        "dimensions": {
            "expression_clarity": {
                "name": "표현 명확성",
                "description": "자신의 수학적 사고를 명확하게 언어로 표현하는 능력",
                "scale": "1-5 (1: 표현 불가, 2: 단편적 표현, 3: 기본 표현, 4: 명확한 표현, 5: 정교한 표현)",
                "initial_value": 3
            },
            "reasoning_quality": {
                "name": "논증 수준",
                "description": "수학적 주장에 대한 근거 제시와 논리적 정당화 능력",
                "scale": "1-5 (1: 근거 없음, 2: 직관적 근거, 3: 부분적 논증, 4: 완전한 논증, 5: 일반화된 증명)",
                "initial_value": 3
            },
            "math_vocabulary_use": {
                "name": "수학 용어 사용",
                "description": "적절한 수학 용어와 기호를 사용하는 능력",
                "scale": "1-5 (1: 일상어만 사용, 2: 기본 용어, 3: 적절한 용어, 4: 풍부한 용어, 5: 전문적 용어)",
                "initial_value": 3
            }
        }
    },
    "emotional_state": {
        "name": "감정 상태 (Emotional State)",
        "description": "학습 과정에서의 정서적 상태와 수학에 대한 태도를 나타내는 모델",
        "dimensions": {
            "engagement": {
                "name": "몰입도",
                "description": "과제에 대한 관심과 적극적 참여 정도",
                "scale": "1-5 (1: 무관심, 2: 소극적, 3: 보통, 4: 적극적, 5: 완전 몰입)",
                "initial_value": 3
            },
            "confidence": {
                "name": "자신감",
                "description": "수학 문제 해결에 대한 자기 효능감",
                "scale": "1-5 (1: 매우 불안, 2: 불안, 3: 보통, 4: 자신감, 5: 높은 자신감)",
                "initial_value": 3
            },
            "frustration": {
                "name": "좌절감",
                "description": "문제 해결 과정에서 느끼는 좌절이나 어려움의 정도",
                "scale": "1-5 (1: 없음, 2: 약간, 3: 보통, 4: 높음, 5: 매우 높음)",
                "initial_value": 2
            }
        }
    },
    "collaboration": {
        "name": "협력 역량 (Collaboration Competency)",
        "description": "그룹 내에서 효과적으로 협력하여 문제를 해결하는 능력",
        "dimensions": {
            "contribution": {
                "name": "기여도",
                "description": "토론과 문제 해결에 능동적으로 기여하는 정도",
                "scale": "1-5 (1: 무참여, 2: 최소 참여, 3: 보통 기여, 4: 적극 기여, 5: 주도적 기여)",
                "initial_value": 3
            },
            "responsiveness": {
                "name": "반응성",
                "description": "동료의 아이디어에 대해 경청하고 반응하는 능력",
                "scale": "1-5 (1: 무반응, 2: 소극적 반응, 3: 기본 반응, 4: 적극적 반응, 5: 건설적 확장)",
                "initial_value": 3
            },
            "help_seeking_giving": {
                "name": "도움 주고받기",
                "description": "필요시 도움을 요청하고 동료를 지원하는 능력",
                "scale": "1-5 (1: 고립, 2: 수동적, 3: 기본 상호작용, 4: 적극적 지원, 5: 상호 촉진)",
                "initial_value": 3
            }
        }
    }
}


def create_initial_learner_model(student_name):
    model = {"student_name": student_name, "models": {}}
    for model_key, model_def in LEARNER_MODEL_SCHEMA.items():
        model["models"][model_key] = {}
        for dim_key, dim_def in model_def["dimensions"].items():
            model["models"][model_key][dim_key] = {
                "value": dim_def["initial_value"],
                "history": [{"task": "초기값", "value": dim_def["initial_value"]}]
            }
    return model


learner_models = {
    "human": create_initial_learner_model("사용자"),
    "ai_1": create_initial_learner_model("AI학생1(민준)"),
    "ai_2": create_initial_learner_model("AI학생2(서연)")
}

print("✅ 학습자 모델 초기화 완료")
print(f"  모델 영역: {[v['name'] for v in LEARNER_MODEL_SCHEMA.values()]}")


In [ ]:
# 셀 3: AI 학생 페르소나 (기본 설정, 추후 세분화)

AI_STUDENTS = {
    "ai_1": {
        "name": "민준",
        "persona": (
            "너는 '민준'이라는 이름의 고등학교 2학년 남학생이야. "
            "수학을 좋아하지만 가끔 실수를 하는 편이야. "
            "적극적으로 의견을 제시하고, 동료의 의견에 관심을 보여. "
            "가끔 틀린 접근을 할 수도 있지만, 다른 사람이 설명해주면 잘 이해해. "
            "반말을 사용하고, 자연스러운 학생 말투로 대화해."
        )
    },
    "ai_2": {
        "name": "서연",
        "persona": (
            "너는 '서연'이라는 이름의 고등학교 2학년 여학생이야. "
            "신중한 성격으로, 답을 말하기 전에 한번 더 생각해보는 편이야. "
            "다른 친구의 풀이를 잘 분석하고, 빠진 부분을 찾아내는 것을 잘해. "
            "질문을 잘 하고, '왜 그렇게 생각했어?'와 같은 메타인지적 질문을 자주 해. "
            "반말을 사용하고, 자연스러운 학생 말투로 대화해."
        )
    }
}

print("✅ AI 학생 페르소나 설정 완료")
print(f"  AI학생1: {AI_STUDENTS['ai_1']['name']} - 적극적, 가끔 실수")
print(f"  AI학생2: {AI_STUDENTS['ai_2']['name']} - 신중, 메타인지적 질문")


In [ ]:
# 셀 4: 예시 태스크 (텍스트 형태, 추후 교체 가능)

TASKS = {
    1: {
        "title": "Task 1: 이차함수의 그래프 해석",
        "content": (
            "다음 이차함수의 그래프를 보고 물음에 답하시오.\n"
            "y = x² - 4x + 3\n\n"
            "(1) 이 함수의 꼭짓점의 좌표를 구하시오.\n"
            "(2) x절편을 구하시오.\n"
            "(3) 이 함수의 그래프가 x축과 만나는 점들 사이의 거리를 구하시오.\n"
            "(4) 그래프의 개형을 설명하시오."
        )
    },
    2: {
        "title": "Task 2: 연립부등식의 활용",
        "content": (
            "한 학생이 문구점에서 연필과 볼펜을 구매하려고 한다.\n"
            "연필 한 자루의 가격은 500원, 볼펜 한 자루의 가격은 1200원이다.\n\n"
            "조건:\n"
            "- 총 예산은 10000원 이하이다.\n"
            "- 연필과 볼펜을 합하여 최소 5자루 이상 사야 한다.\n"
            "- 연필은 최소 2자루 이상 사야 한다.\n\n"
            "(1) 위 조건을 연립부등식으로 나타내시오.\n"
            "(2) 가능한 구매 조합을 모두 구하시오.\n"
            "(3) 가장 많은 필기구를 살 수 있는 조합은 무엇인지 설명하시오."
        )
    },
    3: {
        "title": "Task 3: 확률과 통계적 추론",
        "content": (
            "한 학급 30명의 학생들의 수학 시험 점수가 다음과 같이 분포한다.\n\n"
            "60점대: 3명, 70점대: 8명, 80점대: 12명, 90점대: 7명\n\n"
            "(1) 이 학급의 평균 점수를 구간별 대푯값을 이용하여 추정하시오.\n"
            "(2) 임의로 한 학생을 뽑았을 때, 그 학생이 80점 이상일 확률을 구하시오.\n"
            "(3) 이 분포의 특징을 설명하고, 중앙값은 어느 구간에 있을지 추론하시오."
        )
    },
    4: {
        "title": "Task 4: 도형의 증명",
        "content": (
            "평행사변형 ABCD에서 대각선 AC와 BD가 점 O에서 만난다.\n\n"
            "(1) 삼각형 AOB와 삼각형 COD가 합동임을 증명하시오.\n"
            "(2) 위의 결과를 이용하여 '평행사변형의 두 대각선은 서로를 이등분한다'는 "
            "성질을 증명하시오.\n"
            "(3) 이 성질의 역도 성립하는지 논의하시오."
        )
    }
}

print("✅ 태스크 4개 로드 완료")
for k, v in TASKS.items():
    print(f"  {v['title']}")


## 🔧 핵심 엔진
AI 학생 발화 생성 + 학습자 모델 자동 업데이트


In [ ]:
# 셀 5: CollaborativeLearningSession 클래스

class CollaborativeLearningSession:
    def __init__(self, task_id):
        self.task = TASKS[task_id]
        self.task_id = task_id
        self.conversation_history = []
        self.turn_count = 0

    def _build_ai_response_prompt(self, human_message, responding_ai):
        ai_info = AI_STUDENTS[responding_ai]
        other_ai = "ai_2" if responding_ai == "ai_1" else "ai_1"
        other_name = AI_STUDENTS[other_ai]["name"]

        recent_context = ""
        for msg in self.conversation_history[-10:]:
            recent_context += f"[{msg['speaker']}]: {msg['content']}\n"

        prompt = (
            f"{ai_info['persona']}\n\n"
            f"현재 과제:\n{self.task['content']}\n\n"
            f"지금까지의 대화:\n{recent_context}\n"
            f"사용자가 방금 이렇게 말했어: \"{human_message}\"\n\n"
            f"너({ai_info['name']})의 차례야. "
            f"사용자의 말에 자연스럽게 반응하면서 과제 해결에 기여해. "
            f"2~4문장 정도로 짧게 대답해. "
            f"필요하면 {other_name}에게도 의견을 물어봐."
        )
        return prompt

    def _build_model_update_prompt(self, recent_messages):
        model_state = json.dumps(
            {k: {
                mk: {dk: dv["value"] for dk, dv in mv.items()}
                for mk, mv in v["models"].items()
            } for k, v in learner_models.items()},
            ensure_ascii=False, indent=2
        )

        recent_text = "\n".join(
            [f"[{m['speaker']}]: {m['content']}" for m in recent_messages]
        )

        schema_text = ""
        for mk, mv in LEARNER_MODEL_SCHEMA.items():
            schema_text += f"\n## {mv['name']}\n"
            for dk, dv in mv["dimensions"].items():
                schema_text += f"  - {dv['name']} ({dk}): {dv['scale']}\n"

        prompt = (
            "다음 대화를 분석하여 각 학생의 학습자 모델을 업데이트해줘.\n\n"
            f"=== 학습자 모델 스키마 ===\n{schema_text}\n"
            f"=== 현재 모델 상태 ===\n{model_state}\n\n"
            f"=== 최근 대화 ===\n{recent_text}\n\n"
            "각 학생(human, ai_1, ai_2)에 대해 변경이 필요한 차원만 업데이트해줘.\n"
            "반드시 아래 JSON 형식으로만 응답해:\n"
            '{"updates": [\n'
            '  {"student": "human|ai_1|ai_2", "model": "모델키", "dimension": "차원키", "new_value": 값, "reason": "이유"}\n'
            ']}\n'
            '변경이 없으면 빈 리스트를 반환해: {"updates": []}'
        )
        return prompt

    def get_ai_responses(self, human_message):
        self.conversation_history.append({
            "speaker": "사용자", "content": human_message, "turn": self.turn_count
        })
        self.turn_count += 1
        responses = {}

        # AI학생1(민준)
        prompt_1 = self._build_ai_response_prompt(human_message, "ai_1")
        resp_1 = client.messages.create(
            model=MODEL, max_tokens=500,
            messages=[{"role": "user", "content": prompt_1}]
        )
        ai1_text = resp_1.content[0].text
        self.conversation_history.append({
            "speaker": "민준", "content": ai1_text, "turn": self.turn_count
        })
        self.turn_count += 1
        responses["민준"] = ai1_text

        # AI학생2(서연)
        prompt_2 = self._build_ai_response_prompt(human_message, "ai_2")
        resp_2 = client.messages.create(
            model=MODEL, max_tokens=500,
            messages=[{"role": "user", "content": prompt_2}]
        )
        ai2_text = resp_2.content[0].text
        self.conversation_history.append({
            "speaker": "서연", "content": ai2_text, "turn": self.turn_count
        })
        self.turn_count += 1
        responses["서연"] = ai2_text

        return responses

    def update_learner_models(self):
        recent = self.conversation_history[-6:]
        if not recent:
            return []

        prompt = self._build_model_update_prompt(recent)
        resp = client.messages.create(
            model=MODEL, max_tokens=1000,
            messages=[{"role": "user", "content": prompt}]
        )

        try:
            raw = resp.content[0].text
            start = raw.find("{")
            end = raw.rfind("}") + 1
            if start >= 0 and end > start:
                result = json.loads(raw[start:end])
                updates = result.get("updates", [])
                for u in updates:
                    student = u["student"]
                    model_key = u["model"]
                    dim_key = u["dimension"]
                    new_val = u["new_value"]
                    reason = u.get("reason", "")
                    if (student in learner_models and
                        model_key in learner_models[student]["models"] and
                        dim_key in learner_models[student]["models"][model_key]):
                        learner_models[student]["models"][model_key][dim_key]["value"] = new_val
                        learner_models[student]["models"][model_key][dim_key]["history"].append({
                            "task": f"Task {self.task_id}", "value": new_val, "reason": reason
                        })
                return updates
        except (json.JSONDecodeError, KeyError) as e:
            print(f"  ⚠️ 모델 업데이트 파싱 오류: {e}")
            return []
        return []

print("✅ CollaborativeLearningSession 클래스 정의 완료")


## 📈 시각화


In [ ]:
# 셀 6: 시각화 함수

import matplotlib.pyplot as plt
import matplotlib
import numpy as np

# 한글 폰트 설정 (Colab)
!apt-get -qq install fonts-nanum
import matplotlib.font_manager as fm
font_path = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
fm.fontManager.addfont(font_path)
matplotlib.rcParams['font.family'] = 'NanumGothic'
matplotlib.rcParams['axes.unicode_minus'] = False


def print_learner_model(student_key):
    m = learner_models[student_key]
    print(f"\n{'='*50}")
    print(f"📊 학습자 모델: {m['student_name']}")
    print(f"{'='*50}")
    for model_key, model_def in LEARNER_MODEL_SCHEMA.items():
        print(f"\n📌 {model_def['name']}")
        dims = m["models"][model_key]
        for dim_key, dim_def in model_def["dimensions"].items():
            val = dims[dim_key]["value"]
            if isinstance(val, (int, float)):
                bar = "█" * int(val) + "░" * (5 - int(val))
                print(f"   [{bar}] {dim_def['name']}: {val}/5")
            else:
                print(f"   {dim_def['name']}: {val}")


def print_all_models():
    for key in ["human", "ai_1", "ai_2"]:
        print_learner_model(key)


def print_model_comparison():
    print(f"\n{'='*70}")
    print("📊 학습자 모델 비교표")
    print(f"{'='*70}")
    header = f"{'차원':<20} {'사용자':>8} {'민준':>8} {'서연':>8}"
    print(header)
    print("-" * 50)
    for model_key, model_def in LEARNER_MODEL_SCHEMA.items():
        print(f"\n▸ {model_def['name']}")
        for dim_key, dim_def in model_def["dimensions"].items():
            vals = []
            for sk in ["human", "ai_1", "ai_2"]:
                v = learner_models[sk]["models"][model_key][dim_key]["value"]
                vals.append(v)
            if all(isinstance(v, (int, float)) for v in vals):
                print(f"  {dim_def['name']:<18} {vals[0]:>8} {vals[1]:>8} {vals[2]:>8}")


def plot_learner_radar():
    labels = []
    values = {"human": [], "ai_1": [], "ai_2": []}
    for model_key, model_def in LEARNER_MODEL_SCHEMA.items():
        for dim_key, dim_def in model_def["dimensions"].items():
            sample_val = learner_models["human"]["models"][model_key][dim_key]["value"]
            if isinstance(sample_val, (int, float)):
                labels.append(dim_def["name"])
                for sk in ["human", "ai_1", "ai_2"]:
                    values[sk].append(
                        learner_models[sk]["models"][model_key][dim_key]["value"]
                    )

    N = len(labels)
    angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
    angles += angles[:1]

    fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
    colors = {"human": "#FF6B6B", "ai_1": "#4ECDC4", "ai_2": "#45B7D1"}
    names = {"human": "사용자", "ai_1": "민준", "ai_2": "서연"}

    for sk in ["human", "ai_1", "ai_2"]:
        vals = values[sk] + values[sk][:1]
        ax.plot(angles, vals, 'o-', linewidth=2, label=names[sk], color=colors[sk])
        ax.fill(angles, vals, alpha=0.1, color=colors[sk])

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(labels, size=9)
    ax.set_ylim(0, 5)
    ax.set_yticks([1, 2, 3, 4, 5])
    ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
    ax.set_title("학습자 모델 비교 (레이더 차트)", size=14, fontweight='bold', pad=20)
    plt.tight_layout()
    plt.show()


def plot_model_history(student_key, model_key):
    m = learner_models[student_key]
    model_def = LEARNER_MODEL_SCHEMA[model_key]
    dims = [(k, v) for k, v in model_def["dimensions"].items()
            if isinstance(m["models"][model_key][k]["history"][0]["value"], (int, float))]

    if not dims:
        return

    fig, axes = plt.subplots(1, len(dims), figsize=(4*len(dims), 4))
    if len(dims) == 1:
        axes = [axes]

    for ax, (dim_key, dim_def) in zip(axes, dims):
        history = m["models"][model_key][dim_key]["history"]
        tasks = [h["task"] for h in history]
        vals = [h["value"] for h in history]
        ax.plot(tasks, vals, 'o-', color='#4ECDC4', linewidth=2, markersize=8)
        ax.set_ylim(0, 5.5)
        ax.set_title(dim_def["name"], fontsize=11)
        ax.set_ylabel("수준")
        ax.grid(True, alpha=0.3)

    fig.suptitle(f"{m['student_name']} - {model_def['name']} 변화 추이",
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()


print("✅ 시각화 함수 준비 완료")
# 초기 상태 레이더 차트 미리보기
plot_learner_radar()


## 🚀 세션 실행

아래 셀을 실행하면 대화형 협력학습이 시작됩니다.

**명령어:**
- 메시지 입력 → AI 학생들이 응답
- `/model` → 현재 학습자 모델 출력
- `/compare` → 비교표 출력
- `/radar` → 레이더 차트
- `/history 모델키` → 변화 추이 (cognitive_state, math_communication, emotional_state, collaboration)
- `/quit` → 세션 종료


In [ ]:
# 셀 7: 세션 실행

def run_session(task_id=1):
    session = CollaborativeLearningSession(task_id)

    print(f"\n{'='*60}")
    print(f"🎓 협력학습 세션 시작")
    print(f"{'='*60}")
    print(f"📝 {session.task['title']}")
    print(f"{'─'*60}")
    print(session.task['content'])
    print(f"{'─'*60}")
    print("👥 참여자: 사용자(나), 민준(AI), 서연(AI)")
    print("💡 명령어: /model, /compare, /radar, /history, /quit")
    print(f"{'='*60}\n")

    update_interval = 3
    user_turn = 0

    while True:
        user_input = input("\n🙋 나: ").strip()
        if not user_input:
            continue

        if user_input.startswith("/"):
            cmd = user_input.lower().split()
            if cmd[0] == "/quit":
                print("\n📊 세션 종료. 최종 학습자 모델:")
                print_model_comparison()
                plot_learner_radar()
                break
            elif cmd[0] == "/model":
                print_all_models()
                continue
            elif cmd[0] == "/compare":
                print_model_comparison()
                continue
            elif cmd[0] == "/radar":
                plot_learner_radar()
                continue
            elif cmd[0] == "/history":
                model_key = cmd[1] if len(cmd) > 1 else "cognitive_state"
                for sk in ["human", "ai_1", "ai_2"]:
                    plot_model_history(sk, model_key)
                continue
            else:
                print("  알 수 없는 명령어")
                continue

        print("\n  💭 AI 학생들이 생각하는 중...")
        responses = session.get_ai_responses(user_input)

        for name, text in responses.items():
            print(f"\n🧑‍🎓 {name}: {text}")

        user_turn += 1

        if user_turn % update_interval == 0:
            print("\n  🔄 학습자 모델 업데이트 중...")
            updates = session.update_learner_models()
            if updates:
                print(f"  📊 {len(updates)}개 항목 업데이트:")
                for u in updates:
                    student_name = learner_models[u['student']]['student_name']
                    dim_name = ""
                    for mk, mv in LEARNER_MODEL_SCHEMA.items():
                        if u["dimension"] in mv["dimensions"]:
                            dim_name = mv["dimensions"][u["dimension"]]["name"]
                            break
                    print(f"     {student_name}의 {dim_name}: → {u['new_value']} ({u['reason']})")
            else:
                print("  ✅ 변경 없음")

    return session

# 🎯 Task 번호를 바꿔서 실행하세요 (1~4)
session = run_session(1)
